## 3. XGBoost Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [2]:
import json

from src.dataloader import load_arrays, load_cgm_stats
from src.metrics import evaluate, format_results_table
from src.models.classical_baseline import XGBoostPredictor, HORIZONS

MODEL_NAME = 'XGBoost'
SEED = 42
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / f'xgboost_seed{SEED}_results.json'

In [4]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)

train (83532, 24) val (16739, 24) test (16842, 24)


In [5]:
model = XGBoostPredictor(random_state=SEED).fit(X_train, y_train)

In [8]:
val_results = evaluate(model.predict(X_val), y_val, stats['cgm_mean'], stats['cgm_std'])
test_results = evaluate(model.predict(X_test), y_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation ===')
print(format_results_table(val_results))
print()
print('=== Test ===')
print(format_results_table(test_results))

=== Validation ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.41 |     7.59 |    12.58 |      99.55
           30 |    18.06 |    11.92 |    20.58 |      99.25
           60 |    25.61 |    17.01 |    30.53 |      98.64
           90 |    29.85 |    20.19 |    36.16 |      98.17
          120 |    32.35 |    22.29 |    39.51 |      97.81

=== Test ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.86 |     7.79 |    13.41 |      99.84
           30 |    19.17 |    12.48 |    22.50 |      99.51
           60 |    27.60 |    18.03 |    33.90 |      99.00
           90 |    32.65 |    21.65 |    40.84 |      98.46
          120 |    35.50 |    23.80 |    44.70 |      98.04


In [9]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': SEED,
        'val_results': val_results,
        'test_results': test_results,
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')

saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/xgboost_seed42_results.json
